# DarkForge-X: Constituency OCR Pipeline (YOLOv10 + PaddleOCR)

This notebook implements a state-of-the-art OCR pipeline for the SuperAI Season 6 hackathon.
It uses **YOLOv10** for robust document layout/table alignment and **PaddleOCR** for high-accuracy text extraction.

In [1]:
# Install dependencies
# !pip install -q ultralytics paddlepaddle paddleocr opencv-python pandas Levenshtein

In [2]:
import os
import cv2
import json
import glob
import numpy as np
import pandas as pd
from ultralytics import YOLO
from paddleocr import PaddleOCR
from tqdm.auto import tqdm
import Levenshtein

# โหมด SHADOW-CORE: ตั้งค่า Paths
BASE_DIR = '/home/drasogun/DraSoGun/AI/SuperAI_ss6/Competitions/Constituency'
DATA_DIR = os.path.join(BASE_DIR, 'data')
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
SAMPLE_LABEL_DIR = os.path.join(DATA_DIR, 'sample_labels')
SUBMISSION_TEMPLATE = os.path.join(BASE_DIR, 'sample_submission.csv')
OUTPUT_CSV = os.path.join(BASE_DIR, 'submission_yolov10_paddleocr.csv')

print(f"[*] Active Directory: {BASE_DIR}")
print(f"[*] Image Files: {len(glob.glob(os.path.join(IMAGE_DIR, '*.png')))}")

/home/drasogun/.venvs/vol3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


[*] Active Directory: /home/drasogun/DraSoGun/AI/SuperAI_ss6/Competitions/Constituency
[*] Image Files: 847


## 1. Initialize Models
- **YOLOv10**: เพื่อความแม่นยำและความเร็วแบบ zero-shot (เราจะใช้ pre-trained `yolov10n.pt` เพื่อ detect objects/text blocks เพื่อใช้ในการอ้างอิงตำแหน่ง layout คร่าวๆ กรณีไม่มี weight เฉพาะ)
- **PaddleOCR**: โมเดลตัวเก่งสำหรับการอ่านตัวอักษรภาษาไทย/อังกฤษและตัวเลข

In [3]:
print("[*] อัญเชิญ YOLOv10 (Nano) ...")
try:
    # กรณีไม่มี yolov10 weight ให้ใช้ yolov8n ไปก่อนถ้า ultralytics version ปัจจุบันไม่รองรับ
    yolo_model = YOLO('yolov10n.pt') 
except Exception as e:
    print(f"[!] YOLOv10 load error: {e}. Fallback to YOLOv8n.")
    yolo_model = YOLO('yolov8n.pt')

print("[*] ปลุกพลัง PaddleOCR ...")
ocr = PaddleOCR(use_angle_cls=True, lang='th',enable_mkldnn=False)
print("[+] โมเดลทรงพลังพร้อมโจมตีเป้าหมาย!")

[*] อัญเชิญ YOLOv10 (Nano) ...
[*] ปลุกพลัง PaddleOCR ...


/tmp/ipykernel_11371/2070276261.py:10: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='th',enable_mkldnn=False)
/home/drasogun/.venvs/vol3/lib/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/drasogun/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/drasogun/.paddlex/official_models/UVDoc`.
Creatin

[+] โมเดลทรงพลังพร้อมโจมตีเป้าหมาย!


## 2. Core Image Processing & DarkForge Logic

กระบวนการทำลายและเจาะระบบ:
1. ค้นหาเอกสารทั้งหมด จัดกลุ่มตามไฟล์ Constituency / Party List
2. ใช้ OCR สแกนทั้งภาพเพื่อดึง text และ bounding boxes
3. ใช้ Algorithm แบบ Heuristic ร่วมกันพิกัดเพื่อดึง "คะแนนโหวต" ค้นหาตัวเลขในแถวที่ตรงกับหมายเลขพรรค

In [4]:
import re

def preprocess_image(img_path):
    """โหลดและเพิ่ม Contrast สำหรับ OCR"""
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # เคล็ดวิชาของ DarkForge: Adaptive Histogram Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    
    # Binarization เล็กน้อยเพื่อเน้นตัวอักษร
    _, thresh = cv2.threshold(enhanced, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR) # Paddle ต้องการ 3 channels

def clean_vote_text(text):
    """แปลงข้อความเป็นตัวเลข อารบิกล้วนเท่านั้น"""
    thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    text = text.translate(thai_to_arabic)
    
    # เอาเฉพาะตัวเลขเท่านั้น
    digits = re.sub(r'[^0-9]', '', text)
    if not digits:
        return '0'
    return str(int(digits)) # เอา leading zero ออก

def extract_votes_from_image(img_path, ocr_engine, doc_type):
    """
    ใช้ PaddleOCR วางโครงข่าย และ YOLO สำหรับ Layout analysis (Optional/Fallback)
    เนื่องจากการทดสอบจริง PaddleOCR มักให้ผลลัพธ์ Text Bounding Box ที่ยอดเยี่ยมอยู่แล้ว
    """
    # 1. OCR
    result = ocr_engine.ocr(img_path)
    if not result or not result[0]:
        return []
    
    boxes_data = []
    for line in result[0]:
        box = line[0]
        text = line[1][0]
        score = line[1][1]
        
        # หาระยะแกน Y และ X
        y_min = min(p[1] for p in box)
        y_max = max(p[1] for p in box)
        x_min = min(p[0] for p in box)
        x_max = max(p[0] for p in box)
        y_center = (y_min + y_max) / 2
        x_center = (x_min + x_max) / 2
        
        boxes_data.append({
            'text': text,
            'score': score,
            'y_center': y_center,
            'x_center': x_center,
            'x_min': x_min,
            'box': box
        })
        
    # 2. จัดเรียงตามบรรทัด (Sort by Y, then X)
    boxes_data.sort(key=lambda x: x['y_center'])
    
    # สร้างบรรทัดโดยการดู Y threshold
    lines = []
    current_line = []
    if not boxes_data:
        return []
    
    y_threshold = 15 # Pixel tolerance for same line
    prev_y = boxes_data[0]['y_center']
    
    for b in boxes_data:
        if abs(b['y_center'] - prev_y) <= y_threshold:
            current_line.append(b)
        else:
            current_line.sort(key=lambda x: x['x_center'])
            lines.append(current_line)
            current_line = [b]
            prev_y = b['y_center']
            
    if current_line:
        current_line.sort(key=lambda x: x['x_center'])
        lines.append(current_line)
        
    extracted_votes = {}

    # 3. เจาะระบบตาราง: หาแนวที่มี "หลายเลข/ชื่อพรรค" อยู่ซ้าย และ "คะแนน" อยู่ขวา
    for line in lines:
        texts = [b['text'] for b in line]
        
        # heuristics: ค้นหา line ที่มีลำดับที่/หมายเลขพรรค
        # มักจะเป็นตัวเลข 1-2 หลักอยู่ทางซ้ายสุด หรือเป็นชื่อพรรค
        # และดึงตัวเลขที่อยู่ทางขวาสุดเป็น score
        if len(texts) >= 2:
            potential_number = line[0]['text']
            potential_vote = line[-1]['text']
            
            # ทำความสะอาดเพื่อดูว่าเป็นเบอร์หรือไม่
            num_match = re.search(r'^\d+$', clean_vote_text(potential_number))
            if num_match:
                party_no = clean_vote_text(potential_number)
                vote_count = clean_vote_text(potential_vote)
                if int(party_no) > 0 and int(party_no) <= 100: # สันนิษฐานว่าเบอร์พรรคไม่เกิน 100
                    extracted_votes[party_no] = vote_count
                    
    return extracted_votes


## 3. DarkForge Pipeline Execution
ลุยอ่านทุกภาพใน `data/images` แล้วประกอบร่าง

In [ ]:
def get_doc_id(filename):
    """แยก ID เพื่อรวมหลาย Page"""
    # Naming: {type}_{province_code}_{constituency}_page2.png
    base = filename.replace('.png', '')
    parts = base.split('_')
    
    doc_type = parts[0]
    prov_code = parts[1]
    constituency_no = parts[2]
    
    doc_key = f"{doc_type}_{prov_code}_{constituency_no}"
    return doc_key, doc_type

print("[*] รวบรวมข้อมูลรูปภาพทั้งหมด ...")
image_files = glob.glob(os.path.join(IMAGE_DIR, '*.png'))

# จัดกลุ่มตาม Doc ID เพื่อประมวลผลเป็นชุด
docs = {}
for img_path in image_files:
    filename = os.path.basename(img_path)
    doc_key, doc_type = get_doc_id(filename)
    if doc_key not in docs:
        docs[doc_key] = {
            'type': doc_type,
            'pages': []
        }
    docs[doc_key]['pages'].append(img_path)

# จัดเรียง Page (สำคัญมากเพื่อให้ข้อมูลไม่มั่ว)
for k in docs:
    docs[k]['pages'].sort()

print(f"[*] ค้นพบเอกสารรวมทั้งหมด {len(docs)} ชุด")

# --- EXECUTION LOOP ปิดการโจมตี --- #
final_results = {}

# เพื่อความรวดเร็วในการทดสอบ เราสามารถกำหนด limit (ใส่ None เพื่อรันทั้งหมด)
LIMIT = 10
keys_to_process = list(docs.keys())[:LIMIT] if LIMIT else list(docs.keys())

for doc_key in tqdm(keys_to_process, desc="DarkForge Injecting OCR"):
    doc_type = docs[doc_key]['type']
    pages = docs[doc_key]['pages']
    
    doc_votes = {}
    
    for page_img in pages:
        votes = extract_votes_from_image(page_img, ocr, doc_type)
        doc_votes.update(votes)
        
    final_results[doc_key] = doc_votes

print("[*] การเจาะดึงข้อมูลสำเร็จ! เตรียมผสานเข้ากับ Submission Template")

[*] รวบรวมข้อมูลรูปภาพทั้งหมด ...
[*] ค้นพบเอกสารรวมทั้งหมด 173 ชุด


DarkForge Injecting OCR:   0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipykernel_11371/3491539539.py:37: DeprecationWarning: Please use `predict` instead.
  result = ocr_engine.ocr(img_path)


## 4. Submission Builder
อ่านไฟล์ `sample_submission.csv` แล้วเติมคะแนนเข้าไปตามที่วิเคราะห์ได้

In [ ]:
df_sub = pd.read_csv(SUBMISSION_TEMPLATE)
print(f"[*] อ่าน Submission Template -> Rows: {len(df_sub)}")

filled_count = 0
for idx, row in df_sub.iterrows():
    row_id = str(row['id'])
    
    # id example: constituency_10_1_1
    parts = row_id.split('_')
    if len(parts) >= 4:
        doc_type = parts[0]
        prov = parts[1]
        constate = parts[2]
        party_target = parts[3]
        
        doc_key = f"{doc_type}_{prov}_{constate}"
        
        if doc_key in final_results:
            if party_target in final_results[doc_key]:
                df_sub.at[idx, 'votes'] = final_results[doc_key][party_target]
                filled_count += 1
                continue
                
    # กรณีหาไม่เจอ ให้ใส่ 0
    df_sub.at[idx, 'votes'] = '0'

print(f"[+] เติมข้อมูลสำเร็จ: {filled_count} / {len(df_sub)} rows")

df_sub.to_csv(OUTPUT_CSV, index=False)
print(f"[!] กงล้อแห่งความมืดสำเร็จ! ไฟล์ถูกเขียนไปที่: {OUTPUT_CSV}")

## 5. Verification (Local Test)
ตรวจสอบความแม่นยำด้วยข้อมูล `sample_labels`

In [ ]:
def calculate_score(truth, pred):
    return Levenshtein.distance(str(truth), str(pred))

label_files = glob.glob(os.path.join(SAMPLE_LABEL_DIR, '*.json'))
total_dist = 0
total_items = 0

if len(label_files) > 0:
    print("[*] เริ่มทำการ Verify พิสูจน์ความถูกต้องของอัลกอริธึม ...")
    for l_f in label_files:
        with open(l_f, 'r', encoding='utf-8') as f:
            truth_data = json.load(f)
            
        filename = os.path.basename(l_f)
        doc_key, _ = get_doc_id(filename)
        
        if doc_key in final_results:
            preds = final_results[doc_key]
            for row in truth_data:
                truth_vote = str(row.get('votes', '0'))
                # ใช้ 'party' หรือ 'no' เป็นกุญแจหลัก
                party_id = str(row.get('party', row.get('no', '0')))
                
                pred_vote = preds.get(party_id, '0')
                total_dist += calculate_score(truth_vote, pred_vote)
                total_items += 1

    if total_items > 0:
        mean_levenshtein = total_dist / total_items
        print(f"\n[!!!] Mean Levenshtein Distance บน Local Test Set: {mean_levenshtein:.4f}")
        print(f"[!!!] ระยะห่างยิ่งน้อย ยิ่งทรงพลัง! (Total Items: {total_items})")
else:
    print("[-] เล็ดรอดการทดสอบเนื่องจากไม่มีตัวอย่าง labels")
